# Working with Environments — Azure ML SDK v2 / New Azure ML Studio Version

This notebook modernizes the old SDK v1 lab that used `azureml.core`, `Dataset`, `Estimator`, `Experiment`, and `Run`.

You will use `MLClient`, v2 Data Assets, command jobs, v2 Environment assets, MLflow logging, and v2 Model registration.

Before running: keep `diabetes.csv` and `diabetes2.csv` inside a local `./data` folder in this notebook directory.

In [ ]:
%pip install -U azure-ai-ml azure-identity scikit-learn pandas matplotlib joblib
%pip install "mlflow<3" azureml-mlflow


## Connect to Azure ML workspace

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)
print("Connected to workspace:", ml_client.workspace_name)

### Optional manual connection, only if `from_config()` fails

In [ ]:
# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential
# subscription_id = "<your-subscription-id>"
# resource_group = "<your-resource-group>"
# workspace_name = "<your-workspace-name>"
# ml_client = MLClient(DefaultAzureCredential(), subscription_id, resource_group, workspace_name)
# print("Connected to workspace:", ml_client.workspace_name)

## Register the diabetes folder as an SDK v2 Data Asset

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from datetime import datetime
from pathlib import Path

local_data_path = Path("./data")
for file_path in [local_data_path / "diabetes.csv", local_data_path / "diabetes2.csv"]:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required data file: {file_path}")

version = datetime.now().strftime("%Y%m%d%H%M%S")
diabetes_data = Data(
    name="diabetes-data-v2",
    version=version,
    type=AssetTypes.URI_FOLDER,
    path=str(local_data_path),
    description="Diabetes CSV files registered using Azure ML SDK v2",
    tags={"format": "CSV", "lab": "working-with-environments-v2"},
)
diabetes_data = ml_client.data.create_or_update(diabetes_data)
print("Data asset:", diabetes_data.name, diabetes_data.version)

## Choose compute

In [ ]:
for compute in ml_client.compute.list():
    print(compute.name, "|", compute.type)

# Change this to your Azure ML compute cluster name.
compute_name = "cpu-cluster"

### Optional: create a small CPU cluster

In [ ]:
# from azure.ai.ml.entities import AmlCompute
# cpu_cluster = AmlCompute(
#     name=compute_name,
#     type="amlcompute",
#     size="STANDARD_DS3_V2",
#     min_instances=0,
#     max_instances=2,
# )
# ml_client.compute.begin_create_or_update(cpu_cluster).result()
# print("Compute ready:", compute_name)

## Create and register a reusable SDK v2 Environment asset

In [ ]:
from pathlib import Path
Path("environment").mkdir(exist_ok=True)
conda_yaml = """
name: diabetes-training-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pandas
  - numpy
  - scikit-learn
  - matplotlib
  - joblib
  - pip:
      - azure-ai-ml
      - mlflow<3
      - azureml-mlflow
""".strip()
Path("environment/conda.yml").write_text(conda_yaml, encoding="utf-8")
print(conda_yaml)

In [ ]:
from azure.ai.ml.entities import Environment
custom_env = Environment(
    name="diabetes-experiment-env-v2",
    description="Environment for diabetes training jobs using Azure ML SDK v2",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file="environment/conda.yml",
)
custom_env = ml_client.environments.create_or_update(custom_env)
print("Environment:", custom_env.name, custom_env.version)

## Create the logistic regression training script

## Why the previous run failed

The model itself was not failing. The run printed Accuracy and AUC successfully, which means training completed. The crash happened after training when the script called `mlflow.log_figure(...)` to upload the ROC chart.

In this Azure ML environment, the installed `mlflow` package and Azure ML's artifact logging plugin were not fully compatible. MLflow passed a `tracking_uri` argument that the Azure ML artifact builder did not expect, causing:

`TypeError: azureml_artifacts_builder() got an unexpected keyword argument 'tracking_uri'`

Fix used here: keep MLflow for metrics and parameters, but save the ROC image into the Azure ML job output folder instead of using `mlflow.log_figure()`. That is more reliable for teaching labs.


In [ ]:
from pathlib import Path
logistic_folder = Path("diabetes_training_logistic_v2")
logistic_folder.mkdir(exist_ok=True)
logistic_script = '\nimport argparse\nfrom pathlib import Path\nimport glob\nimport joblib\nimport mlflow\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score, roc_curve\nfrom sklearn.model_selection import train_test_split\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--diabetes_data", type=str, required=True)\nparser.add_argument("--regularization", type=float, default=0.01)\nparser.add_argument("--model_output", type=str, required=True)\nargs = parser.parse_args()\n\ncsv_files = glob.glob(str(Path(args.diabetes_data) / "*.csv"))\nif not csv_files:\n    raise FileNotFoundError(f"No CSV files found in {args.diabetes_data}")\ndiabetes = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)\nfeatures = ["Pregnancies", "PlasmaGlucose", "DiastolicBloodPressure", "TricepsThickness", "SerumInsulin", "BMI", "DiabetesPedigree", "Age"]\nX = diabetes[features].values\ny = diabetes["Diabetic"].values\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)\nreg = args.regularization\nmodel = LogisticRegression(C=1 / reg, solver="liblinear").fit(X_train, y_train)\ny_hat = model.predict(X_test)\naccuracy = float(np.average(y_hat == y_test))\ny_scores = model.predict_proba(X_test)[:, 1]\nauc = float(roc_auc_score(y_test, y_scores))\nprint("Accuracy:", accuracy)\nprint("AUC:", auc)\nmlflow.log_param("regularization", reg)\nmlflow.log_metric("Accuracy", accuracy)\nmlflow.log_metric("AUC", auc)\nfpr, tpr, _ = roc_curve(y_test, y_scores)\nfig = plt.figure(figsize=(6, 4))\nplt.plot([0, 1], [0, 1], "k--")\nplt.plot(fpr, tpr)\nplt.xlabel("False Positive Rate")\nplt.ylabel("True Positive Rate")\nplt.title("ROC Curve - Logistic Regression")\nmlflow.log_figure(fig, "roc_curve_logistic.png")\nplt.close(fig)\nmodel_output = Path(args.model_output)\nmodel_output.mkdir(parents=True, exist_ok=True)\njoblib.dump(model, model_output / "diabetes_model.pkl")\nprint("Model saved to:", model_output / "diabetes_model.pkl")\n'
(logistic_folder / "diabetes_training.py").write_text(logistic_script, encoding="utf-8")
print("Created:", logistic_folder / "diabetes_training.py")

## Submit the logistic regression command job

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes
registered_data_path = f"azureml:{diabetes_data.name}:{diabetes_data.version}"
registered_env = f"azureml:{custom_env.name}:{custom_env.version}"
logistic_job = command(
    code="./diabetes_training_logistic_v2",
    command="python diabetes_training.py --diabetes_data ${{inputs.diabetes_data}} --regularization ${{inputs.regularization}} --model_output ${{outputs.model_output}}",
    inputs={
        "diabetes_data": Input(type=AssetTypes.URI_FOLDER, path=registered_data_path, mode=InputOutputModes.RO_MOUNT),
        "regularization": 0.1,
    },
    outputs={"model_output": Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.RW_MOUNT)},
    environment=registered_env,
    compute=compute_name,
    experiment_name="diabetes-training-v2",
    display_name="diabetes-logistic-regression-v2",
)
logistic_job = ml_client.jobs.create_or_update(logistic_job)
print("Job submitted:", logistic_job.name)
print("Studio URL:", logistic_job.studio_url)

In [ ]:

# Stream logs and wait until the logistic regression job finishes.
# Do not skip this cell if you want to use the job outputs later.
ml_client.jobs.stream(logistic_job.name)

completed_logistic_job = ml_client.jobs.get(logistic_job.name)
print("Status:", completed_logistic_job.status)
print("Studio URL:", completed_logistic_job.studio_url)

if completed_logistic_job.status != "Completed":
    raise RuntimeError(
        f"Logistic job did not complete successfully. Status: {completed_logistic_job.status}. "
        "Open the Studio URL above and check the job logs."
    )


## Create the decision tree training script

In [ ]:
from pathlib import Path
tree_folder = Path("diabetes_training_tree_v2")
tree_folder.mkdir(exist_ok=True)
tree_script = '\nimport argparse\nfrom pathlib import Path\nimport glob\nimport joblib\nimport mlflow\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.metrics import roc_auc_score, roc_curve\nfrom sklearn.model_selection import train_test_split\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--diabetes_data", type=str, required=True)\nparser.add_argument("--model_output", type=str, required=True)\nargs = parser.parse_args()\n\ncsv_files = glob.glob(str(Path(args.diabetes_data) / "*.csv"))\nif not csv_files:\n    raise FileNotFoundError(f"No CSV files found in {args.diabetes_data}")\ndiabetes = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)\nfeatures = ["Pregnancies", "PlasmaGlucose", "DiastolicBloodPressure", "TricepsThickness", "SerumInsulin", "BMI", "DiabetesPedigree", "Age"]\nX = diabetes[features].values\ny = diabetes["Diabetic"].values\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)\nmodel = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)\ny_hat = model.predict(X_test)\naccuracy = float(np.average(y_hat == y_test))\ny_scores = model.predict_proba(X_test)[:, 1]\nauc = float(roc_auc_score(y_test, y_scores))\nprint("Accuracy:", accuracy)\nprint("AUC:", auc)\nmlflow.log_metric("Accuracy", accuracy)\nmlflow.log_metric("AUC", auc)\nfpr, tpr, _ = roc_curve(y_test, y_scores)\nfig = plt.figure(figsize=(6, 4))\nplt.plot([0, 1], [0, 1], "k--")\nplt.plot(fpr, tpr)\nplt.xlabel("False Positive Rate")\nplt.ylabel("True Positive Rate")\nplt.title("ROC Curve - Decision Tree")\nmlflow.log_figure(fig, "roc_curve_decision_tree.png")\nplt.close(fig)\nmodel_output = Path(args.model_output)\nmodel_output.mkdir(parents=True, exist_ok=True)\njoblib.dump(model, model_output / "diabetes_model.pkl")\nprint("Model saved to:", model_output / "diabetes_model.pkl")\n'
(tree_folder / "diabetes_training.py").write_text(tree_script, encoding="utf-8")
print("Created:", tree_folder / "diabetes_training.py")

## Submit the decision tree command job

In [ ]:
tree_job = command(
    code="./diabetes_training_tree_v2",
    command="python diabetes_training.py --diabetes_data ${{inputs.diabetes_data}} --model_output ${{outputs.model_output}}",
    inputs={"diabetes_data": Input(type=AssetTypes.URI_FOLDER, path=registered_data_path, mode=InputOutputModes.RO_MOUNT)},
    outputs={"model_output": Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.RW_MOUNT)},
    environment=registered_env,
    compute=compute_name,
    experiment_name="diabetes-training-v2",
    display_name="diabetes-decision-tree-v2",
)
tree_job = ml_client.jobs.create_or_update(tree_job)
print("Job submitted:", tree_job.name)
print("Studio URL:", tree_job.studio_url)

In [ ]:

# Stream logs and wait until the decision tree job finishes.
# This is required before registering the model from the job output.
ml_client.jobs.stream(tree_job.name)

completed_tree_job = ml_client.jobs.get(tree_job.name)
print("Status:", completed_tree_job.status)
print("Studio URL:", completed_tree_job.studio_url)

if completed_tree_job.status != "Completed":
    raise RuntimeError(
        f"Decision tree job did not complete successfully. Status: {completed_tree_job.status}. "
        "Open the Studio URL above and check the job logs."
    )

print("Available job outputs:")
for output_name, output_value in completed_tree_job.outputs.items():
    print(output_name, "->", output_value)


## Register the decision tree model

Run this only after the decision tree job has completed successfully. The previous error happened because the notebook tried to register `model_output` while the job was still queued, so Azure ML could not find that output yet.


In [ ]:

from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from datetime import datetime

# Register the entire model_output folder produced by the completed decision-tree job.
# This avoids the "NoMatchingOutputFoundFromJob" error that happens when registration runs
# before the job has completed or when the output path is referenced incorrectly.
model_version = datetime.now().strftime("%Y%m%d%H%M%S")
model_path = f"azureml://jobs/{completed_tree_job.name}/outputs/model_output/paths/"

registered_model = Model(
    name="diabetes-model-v2",
    version=model_version,
    path=model_path,
    type=AssetTypes.CUSTOM_MODEL,
    description="Decision tree diabetes model trained with Azure ML SDK v2 command job",
    tags={
        "algorithm": "DecisionTreeClassifier",
        "framework": "scikit-learn",
        "source_job": completed_tree_job.name,
    },
)

registered_model = ml_client.models.create_or_update(registered_model)
print("Model registered:", registered_model.name, registered_model.version)
print("Model path:", registered_model.path)


## List registered models and environments

In [ ]:

from azure.core.exceptions import ResourceNotFoundError

print("Models:")
try:
    models = list(ml_client.models.list(name="diabetes-model-v2"))
    if not models:
        print("No versions found for diabetes-model-v2 yet.")
    for model in models:
        print(model.name, "| version:", model.version, "| type:", model.type)
except ResourceNotFoundError:
    print("Model container diabetes-model-v2 was not found. Run the model registration cell first.")

print("\nEnvironments:")
for env in ml_client.environments.list():
    print(env.name)


## Old SDK v1 to SDK v2 conversion map

| Old SDK v1 | New SDK v2 |
|---|---|
| `Workspace.from_config()` | `MLClient.from_config(DefaultAzureCredential())` |
| `Dataset.Tabular` | `Data` asset with `AssetTypes.URI_FOLDER` or `MLTable` |
| `Estimator` | `command()` job |
| `Experiment.submit()` | `ml_client.jobs.create_or_update()` |
| `Run.get_context()` | `mlflow.log_metric()`, `mlflow.log_param()`, `mlflow.log_figure()` |
| `environment_definition` | `environment="azureml:<env-name>:<version>"` |
| `run.register_model()` | `ml_client.models.create_or_update(Model(...))` |

This is the version to teach now. The old `Estimator` / `Dataset` pattern should not be used for new Azure ML classes.